In [1]:
from langchain_ollama import OllamaLLM
import pandas as pd
import numpy as np
import re
from cleantext import clean
Words2Rmv = ['xxx']

In [2]:
llm = OllamaLLM(model="financial_analyser_llama2:latest")
llm.invoke("Who is leading using AI to fight financial crime?")

"\nAs a financial assistant, I can tell you that several organizations and institutions are leveraging artificial intelligence (AI) to combat financial crime. Here are some examples:\n\n1. FinCEN: The Financial Crimes Enforcement Network (FinCEN) is a bureau of the U.S. Department of the Treasury that collects and analyzes financial data to identify and prevent money laundering and other financial crimes. FinCEN uses AI and machine learning algorithms to analyze large volumes of financial data, identify patterns and anomalies, and identify potential cases of financial crime.\n2. SWIFT: The Society for Worldwide Interbank Financial Telecommunication (SWIFT) is a global financial messaging network that connects banks and other financial institutions around the world. SWIFT uses AI and machine learning to detect and prevent fraudulent transactions, such as money laundering and terrorist financing.\n3. UK Financial Intelligence Unit (FIU): The FIU is a unit of the UK's National Crime Agenc

In [15]:
def read_data():
    TransDF = pd.read_csv('sample.csv')
    TransDF['Amount'] = pd.to_numeric(TransDF['Amount'], errors='coerce').fillna(0).astype(int)    
    return TransDF

AllTransDF = read_data()

#make sure the data has the right data type
print(AllTransDF.dtypes)

Date             str
Amount         int64
Transaction      str
dtype: object


In [16]:
AllTransDF.head()

,Date,Amount,Transaction
0,27/03/2025,-288,YARRAVALLEYWATER YVOW
1,11/06/2025,-40,TARGET 5144
2,05/09/2024,-3,EFTPOS PAYPAL * SYDNEY AU
3,02/10/2025,-197,BARWON WATER
4,22/07/2024,-1500,VIRGIN


In [17]:
def transform_data(AllTransDF):
    
    pd.set_option('display.max_colwidth', None)
    #remove unwanted words
    AllTransDF['CleanTransaction'] = AllTransDF['Transaction'].apply(
        lambda x: ' '.join([word for word in x.split() 
                            if word.lower() not in Words2Rmv])
    )
    
    #perform some simple text cleaning using cleantext
    AllTransDF['Transaction'] = AllTransDF['CleanTransaction'].apply(lambda x: clean(
        x,
        clean_all=True,
        extra_spaces=True,
        numbers=True,
        punct=True
        ))
    
    AllTransDF = AllTransDF.drop("CleanTransaction", axis=1)
    AllTransDF['Income/Expense'] = np.where(AllTransDF['Amount']>0, 'Income','Expense')
    AllTransDF['Transaction']=AllTransDF['Transaction'].str.title()
   
    return (AllTransDF)

TransformedDF = transform_data(AllTransDF)

In [18]:
UniqueTransactions=TransformedDF['Transaction'].unique()
len(UniqueTransactions)
UniqueTransactions

<StringArray>
[   'Yarravalleywater Yvow',                   'Target',
 'Eftpos Paypal  Sydney Au',             'Barwon Water',
                   'Virgin',  'Woolworthsgreat Western',
     'Tong Li Supermarkets',    'From Metro Art Galary',
     'Yellow Bird Painting',              'Aldi Mobile',
  'Transfer From Macquarie',   'Payment To Credit Card',
      'Just Ripe Northland', 'Woolworthswestfield Camp',
              'Aldi Stores',    'Body Sculpting Clinic',
                'Sushi Hub', 'Reversal Of Account Fees',
        'Les Family Clinic',           'Payment To Fmc',
    'Sumbul Meats  Poultry',    'Aust Hlth Mgmt Grp Pl',
        'Transfer From Cba',                    'Coles',
       'Velocity Australia',                    'Daiso',
      'Ezymart Central Stn',     'Agodacom Aluna Hotel',
      'Soul Origins Mascot',         'Uber Trip Sydney',
      'Yummy Saigon Summer',      'Paid To Credit Card']
Length: 32, dtype: str

In [19]:
response = llm.invoke('can you add a proper category to the following items. Do not apply reasonong. ' \
'Responde with category. For example: Coles Express Bread - groceries, Big Apple transaction - groceries' + 
', '.join(UniqueTransactions[1:10]))
response = response.split('\n')
response

['',
 'Sure! Here are the categories I would suggest for each of the items you listed:',
 '',
 '1. Coles Express Bread - Groceries',
 '2. Big Apple transaction - Groceries',
 '3. Eftpos Paypal  Sydney Au - Utilities (Eftpos fees)',
 '4. Barwon Water - Utilities (Water bill)',
 '5. Virgin - Communications (Mobile phone bill)',
 '6. Woolworths Great Western - Groceries',
 '7. From Metro Art Galary - Entertainment (Art supply purchase)',
 '8. Yellow Bird Painting - Entertainment (Art supply purchase)',
 '9. Aldi Mobile - Utilities (Phone bill)']

In [20]:
# Generate an arrary of sequence of 10s and append the final boundary index
IndexTransList = (
    np.append(np.arange(0, len(UniqueTransactions), 10), 
              len(UniqueTransactions)).tolist()
)

IndexTransList


[0, 10, 20, 30, 32]

In [22]:
# Validate response 's format
from pydantic import BaseModel, field_validator
from typing import List

class ResponseChecks(BaseModel):
    data: List[str]
    @field_validator('data')
    def check_length(cls, value):
        for item in value:
            if len(item) >0:
                assert "-" in item, f"Item '{item}' does not contain a hyphen"
#Pass asertion
ResponseChecks(data=['Hello World - greeting', 'banana cake - fruit'])
                    
    

ResponseChecks(data=None)

In [10]:
#if the check will fail
ResponseChecks(data=['Hello-World', 'banana fruit'])

ValidationError: 1 validation error for ResponseChecks
data
  Assertion failed, Item 'banana fruit' does not contain a hyphen [type=assertion_error, input_value=['Hello-World', 'banana fruit'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/assertion_error

In [23]:
def categorise_transactions(transactions, llm):
    response = llm.invoke("Can you add an appropriate category to the following expenses. For example: " \
    "Best Baking Syndey Metro - food, oil paint canvas - creative, uberdrive int aiport - transport, utility Sept - utility. " \
    "Categories should be less than 4 characters. " + transactions)
    response = response.split('\n')
    
    #removing the explaination lines at the beginning and end of the response
    TransIndexes = [index for index in range(len(response)) if response[index] == '']
    if len(TransIndexes) == 1:
        response = response[(TransIndexes[0] + 1):]
    else:
        response = response[(TransIndexes[0] + 1) : TransIndexes[1]]

    #validate response for '-' to separate transaction and category
    ResponseChecks(data = response)
    
    # assigned the response to a dataframe
    CategoriesDF = pd.DataFrame({'Transaction - Category': response})
    CategoriesDF[['Transaction', 'Category']] = CategoriesDF['Transaction - Category'].str.split(' - ', expand=True)

    return CategoriesDF


In [24]:
#test the categorise_transactions function
TestTrans = categorise_transactions('Best Baking Syndey Metro, oil paint canvas , uberdrive int aiport, utility Sept',llm)
TestTrans

,Transaction - Category,Transaction,Category
0,* Best Baking Sydney Metro - Food,* Best Baking Sydney Metro,Food
1,* Oil Paint Canvas - Creative,* Oil Paint Canvas,Creative
2,* Uber Drive Int Airport - Transport,* Uber Drive Int Airport,Transport
3,* Utility Sept - Utilities,* Utility Sept,Utilities


In [35]:
def process_categories(UniqueTransactions, IndexTransList, llm):
  
    #initialise a new empty dataframe
    CategoriesDF = pd.DataFrame()
    #set a max tries if LLM output incorrect format. Rule set '-' between transaction and category
    MaxTries = 5

    #loop through the unique transactions and get the block of categories using index list built before
    for i in range(0,len(IndexTransList)-1):
        #extract transaction belong to current block
        BlockTrans= UniqueTransactions[IndexTransList[i]:IndexTransList[i+1]]
        #Separate each transaction with a comma before fit them to LLM
        BlockTrans = ', '.join(BlockTrans)
        # retry loop when LLM return output of incorrect format
        for j in range(MaxTries):
            try:
                #call llm to classify the block of transactions
                Categories = categorise_transactions(BlockTrans, llm)
                #Append the output to the dataframe.
                CategoriesDF = pd.concat([CategoriesDF, Categories], ignore_index=True)                                 
            except:
                if j < MaxTries:
                    continue
                else:
                    raise Exception (f"Failed to categorise transactions after {MaxTries} attempts.")
            break
    return CategoriesDF
#process all uniques categories
CategoriesDF = process_categories(UniqueTransactions, IndexTransList, llm)
CategoriesDF

,Transaction - Category,Transaction,Category
0,1. Best Baking Sydney Metro - Food,1. Best Baking Sydney Metro,Food
1,2. Oil paint canvas - Creative,2. Oil paint canvas,Creative
2,3. Uber drive to airport - Transport,3. Uber drive to airport,Transport
3,4. Utility Sept - Utility,4. Utility Sept,Utility
4,5. Yarravalleywater Yvow - Utility,5. Yarravalleywater Yvow,Utility
5,6. Target - Shopping,6. Target,Shopping
6,7. Eftpos Paypal Sydney Au - Online Spending,7. Eftpos Paypal Sydney Au,Online Spending
7,8. Barwon Water - Utility,8. Barwon Water,Utility
8,9. Virgin - Transport,9. Virgin,Transport
9,10. Woolworths great Western - Groceries,10. Woolworths great Western,Groceries


In [36]:
#remove unwanted characters leading each transaction
CategoriesDF['Transaction'] = CategoriesDF['Transaction'].str.replace(r'^(\d+\.|\*)\s+', '', regex=True).str.strip()
CategoriesDF['Category']=CategoriesDF['Category'].str.title()
CategoriesDF


,Transaction - Category,Transaction,Category
0,1. Best Baking Sydney Metro - Food,Best Baking Sydney Metro,Food
1,2. Oil paint canvas - Creative,Oil paint canvas,Creative
2,3. Uber drive to airport - Transport,Uber drive to airport,Transport
3,4. Utility Sept - Utility,Utility Sept,Utility
4,5. Yarravalleywater Yvow - Utility,Yarravalleywater Yvow,Utility
5,6. Target - Shopping,Target,Shopping
6,7. Eftpos Paypal Sydney Au - Online Spending,Eftpos Paypal Sydney Au,Online Spending
7,8. Barwon Water - Utility,Barwon Water,Utility
8,9. Virgin - Transport,Virgin,Transport
9,10. Woolworths great Western - Groceries,Woolworths great Western,Groceries


In [55]:
def normalise_transaction(value):
    if pd.isna(value):
        return ""
    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()

SourceDF = TransformedDF.copy()
SourceDF = SourceDF.rename(columns={"Transaction": "SrcTransaction"})

CategoriesDF = CategoriesDF.copy()
CategoriesDF["Transaction"] = CategoriesDF["Transaction"].fillna("").astype(str).str.strip()
CategoriesDF["Transaction"] = CategoriesDF["Transaction"].str.replace(r"^(\d+\.|\*)\s+", "", regex=True)
CategoriesDF["JoinKey"] = CategoriesDF["Transaction"].apply(normalise_transaction)
CategoriesDF = CategoriesDF.drop_duplicates(subset=["JoinKey"], keep="first")

SourceDF["JoinKey"] = SourceDF["SrcTransaction"].apply(normalise_transaction)

CategorisedTrans = pd.merge(
    SourceDF,
    CategoriesDF[["JoinKey", "Category"]],
    on="JoinKey",
    how="left"
).drop(columns=["JoinKey"])

CategorisedTrans["Category"] = CategorisedTrans["Category"].fillna("Uncategorised")
CategorisedTrans


,Date,Amount,SrcTransaction,Income/Expense,Category
0,27/03/2025,-288,Yarravalleywater Yvow,Expense,Utility
1,11/06/2025,-40,Target,Expense,Shopping
2,05/09/2024,-3,Eftpos Paypal Sydney Au,Expense,Online Spending
3,02/10/2025,-197,Barwon Water,Expense,Utility
4,22/07/2024,-1500,Virgin,Expense,Transport
5,27/01/2026,-21,Woolworthsgreat Western,Expense,Uncategorised
6,15/01/2025,-62,Tong Li Supermarkets,Expense,Groceries
7,19/06/2026,2000,From Metro Art Galary,Income,Uncategorised
8,11/11/2024,167,Yellow Bird Painting,Income,Creative
9,01/12/2025,-33,Aldi Mobile,Expense,Online Spending


In [42]:
CategorisedTrans = CategorisedTrans.dropna()
CategorisedTrans

,Date,Amount,SrcTransaction,Income/Expense,Category
0,27/03/2025,-288,Yarravalleywater Yvow,Expense,Utility
1,11/06/2025,-40,Target,Expense,Shopping
2,05/09/2024,-3,Eftpos Paypal Sydney Au,Expense,Online Spending
3,02/10/2025,-197,Barwon Water,Expense,Utility
4,22/07/2024,-1500,Virgin,Expense,Transport
6,15/01/2025,-62,Tong Li Supermarkets,Expense,Groceries
8,11/11/2024,167,Yellow Bird Painting,Income,Creative
9,01/12/2025,-33,Aldi Mobile,Expense,Online Spending
10,04/11/2024,2067,Transfer From Macquarie,Income,Transfer
11,17/12/2024,-422,Payment To Credit Card,Expense,Payment


In [43]:
UniqCategories = CategorisedTrans['Category'].unique()
UniqCategories

<StringArray>
[        'Utility',        'Shopping', 'Online Spending',       'Transport',
       'Groceries',        'Creative',        'Transfer',         'Payment',
         'Grocery',          'Health',          'Dining',            'Misc',
          'Travel']
Length: 13, dtype: str

In [ ]:
def regroup_category():
    # 2. Define a lookup dictionary
    LookupDic = {
        "Food": "Dining & Leisure",
        "Entertainment": "Dining & Leisure",
        "Groceries": "Supplies & Essentials",
        "Grocery": "Supplies & Essentials",
        "Shopping": "Shopping & Retails",
        "Clothing": "Shopping & Retails",
        "Home": "Shopping & Retails",
        "Utilities": "Utility"
    }
    CategorisedTrans['RefineCategory'] = CategorisedTrans['Category'].map(LookupDic).fillna(CategorisedTrans['Category'])
    return CategorisedTrans

CategorisedTrans = regroup_category()
CategorisedTrans

,Date,Amount,SrcTransaction,Income/Expense,Category,RefineCategory
0,27/03/2025,-288,Yarravalleywater Yvow,Expense,Utility,Utility
1,11/06/2025,-40,Target,Expense,Shopping,Shopping & Retails
2,05/09/2024,-3,Eftpos Paypal Sydney Au,Expense,Online Spending,Online Spending
3,02/10/2025,-197,Barwon Water,Expense,Utility,Utility
4,22/07/2024,-1500,Virgin,Expense,Transport,Transport
5,27/01/2026,-21,Woolworthsgreat Western,Expense,Uncategorised,Uncategorised
6,15/01/2025,-62,Tong Li Supermarkets,Expense,Groceries,Supplies & Essentials
7,19/06/2026,2000,From Metro Art Galary,Income,Uncategorised,Uncategorised
8,11/11/2024,167,Yellow Bird Painting,Income,Creative,Creative
9,01/12/2025,-33,Aldi Mobile,Expense,Online Spending,Online Spending
